In [7]:
import sys
sys.path.append("../../")  # allow imports from grandparent directory
from src.utils.generate_hamiltonians import calculate_minimum_evolution_time, create_h2_minimal_basis_hamiltonian, generate_ising_hamiltonian
from src.utils.memory_profiling_utils import *
from qiskit.quantum_info import SparsePauliOp
import itertools

In [5]:
# ════════════════════════════════════════════════════════════════════════════
#  Parameter grid and constants
# ════════════════════════════════════════════════════════════════════════════
NUM_SYSTEM_QUBITS  = 4
ISING_J, ISING_G = 1.2, 1
PLACEHOLDER = "_I_"

H_H2 = create_h2_minimal_basis_hamiltonian()

HAMILTONIANS_TO_TEST: dict[str, SparsePauliOp] = {
    # "exact_ham_exact_qdritf" : SparsePauliOp(data="ZZZZ", coeffs=np.pi / 4),
    "H2_minimal_basis": H_H2,
    "H_ising" : generate_ising_hamiltonian(num_qubits=NUM_SYSTEM_QUBITS, J=ISING_J * 0.5, g=ISING_G * 0.5) 
}

NUM_ANCILLA  = [13]  # number of ancilla qubits

# chebyshev_nodes = np.array(chebyshev_nodes(10))
# scaled_nodes_pos = 0.000001 + (0.1 - 0.000001) * chebyshev_nodes[:5]

qpe_resolution_limits = calculate_minimum_evolution_time(hamiltonians=HAMILTONIANS_TO_TEST, m=min(NUM_ANCILLA))
print(qpe_resolution_limits)
t_min_global = max(qpe_resolution_limits.values())
lower_bound = max(1e-10, t_min_global * 0.9)  # Don't go below 90% of t_min
upper_bound = min(1e1, t_min_global * 1000)    # Don't exceed 100× t_min
TIMES = np.logspace(np.log2(lower_bound), np.log2(upper_bound), base=2, num=12)
NUM_QDRIFT_SEGMENTS_PER_CHANNEL_SAMPLE  = [1]
RANDOM_CIRCUITS_PER_DATAPOINT = [100]
SHOTS_PER_CIRCUIT = [1, 100]
REPORT_PROTOCOL_RESULTS_FROM_ANY_RANDOM_CIRCUIT = [{"group": True, "group_by": "median"}]
REPLICATION_SEEDS = [42] # the same seed is used for all circuits in one data point. if more than 1 seed is given, the number of circuits is multiplied by the number of seeds.
ESTIMATE_GROUND_STATE = [False]  # whether to estimate the smallest eigenvalue (ground state). If False we pick the largest eigenvalue (excited state).

{'H2_minimal_basis': 0.0067268038747453915, 'H_ising': 0.005267186425494078}


In [9]:
# full Cartesian product of all sweep parameters
grid = itertools.product(
    HAMILTONIANS_TO_TEST.keys(),
    NUM_ANCILLA,
    TIMES,
    NUM_QDRIFT_SEGMENTS_PER_CHANNEL_SAMPLE,
    REPLICATION_SEEDS,            # outer repetition
    RANDOM_CIRCUITS_PER_DATAPOINT,
    SHOTS_PER_CIRCUIT,
    ESTIMATE_GROUND_STATE,         # whether to estimate the ground state,
    REPORT_PROTOCOL_RESULTS_FROM_ANY_RANDOM_CIRCUIT
)

# serialise each tuple into a plain dict for _run
cfgs = [dict(ham              = g[0],
                anc              = g[1],
                time             = float(g[2]),
                segments         = g[3],
                replication_seed = g[4],
                circuits         = g[5],
                shots            = g[6],
                ground_state     = g[7],
                trajectory_report_protocol = g[8]
                )
        for g in grid]
print(f"Total number of data points to be collected: {len(cfgs)}")

Total number of data points to be collected: 48


In [14]:
plan_parameter_sweep(hamiltonians=HAMILTONIANS_TO_TEST,
                     num_system_qubits=NUM_SYSTEM_QUBITS,
                     num_ancilla_options=NUM_ANCILLA,
                     evolution_times=TIMES,
                     shots_options=SHOTS_PER_CIRCUIT,
                     num_circuits_per_point=RANDOM_CIRCUITS_PER_DATAPOINT,
                     target_precision=0.01,
                     simulator_type="matrix_product_state"
                     )

Planning parameter sweep with 13.6 GB available memory
Simulator type: matrix_product_state

=== Analyzing H2_minimal_basis ===
Number of terms: 15
Total strength (α): 2.698
Coefficient range: 0.045322 to 0.812610
Peak estimated memory: 308317.7 MB (301.09 GB)
Analyzed 24 configurations
Feasible configurations: 20
Infeasible configurations: 4
Most memory-efficient config: 13 ancilla, t=0.006, 1 shots → 959.0 MB

=== Analyzing H_ising ===
Number of terms: 8
Total strength (α): 4.400
Coefficient range: 0.500000 to 0.600000
Peak estimated memory: 1924296.4 MB (1879.20 GB)
Analyzed 24 configurations
Feasible configurations: 18
Infeasible configurations: 6
Most memory-efficient config: 13 ancilla, t=0.006, 1 shots → 959.0 MB

=== Overall Analysis ===
Total feasible configurations: 38
Total infeasible configurations: 10
Feasibility rate: 79.2%

Top infeasibility reasons:
  Exceeds available memory: 10 configurations


{'system_info': {'platform': 'Windows-10-10.0.19045-SP0',
  'python_version': '3.13.1',
  'cpu_count': 12,
  'total_memory_gb': 23.825584411621094,
  'available_memory_gb': 13.646778106689453},
 'hamiltonian_analysis': {'H2_minimal_basis': {'num_terms': 15,
   'total_strength': np.float64(2.6976929999999997),
   'coefficients': [(-0.8126100000000002+0j),
    (0.171201+0j),
    (0.16862325+0j),
    (-0.22279650000000004+0j),
    (0.171201+0j),
    (0.12054625+0j),
    (0.17434925+0j),
    (0.04532175+0j),
    (0.04532175+0j),
    (0.165868+0j),
    (0.12054625+0j),
    (-0.22279650000000004+0j),
    (0.04532175+0j),
    (0.04532175+0j),
    (0.165868+0j)],
   'max_coefficient': 0.8126100000000002,
   'min_coefficient': 0.04532175},
  'H_ising': {'num_terms': 8,
   'total_strength': np.float64(4.4),
   'coefficients': [(-0.6+0j),
    (-0.6+0j),
    (-0.6+0j),
    (-0.6+0j),
    (-0.5+0j),
    (-0.5+0j),
    (-0.5+0j),
    (-0.5+0j)],
   'max_coefficient': 0.6,
   'min_coefficient': 0.5}}